# 01. Разведочный анализ данных (EDA)

**Датасет:** Telco Customer Churn (IBM, открытый).
**Цель:** понять структуру данных, баланс классов, пропуски, распределения признаков перед построением моделей.

Целевая переменная — `Churn` (Yes/No → 1/0): ушёл ли клиент в течение последнего месяца.

In [ ]:
import sys
from pathlib import Path

# чтобы импортировать из src/
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load import load_raw, load_clean

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Загрузка сырых данных

In [ ]:
raw = load_raw(PROJECT_ROOT / 'data' / 'raw' / 'telco_churn.csv')
print('Размер:', raw.shape)
raw.head()

In [ ]:
raw.info()

**Наблюдение:** колонка `TotalCharges` распознаётся как `object` — внутри есть пробелы для клиентов с `tenure=0`. Это нужно обработать (см. `src/data/load.py::clean`).

In [ ]:
# проверка проблемных значений TotalCharges
mask = pd.to_numeric(raw['TotalCharges'], errors='coerce').isna()
print('Строк с некорректным TotalCharges:', mask.sum())
raw.loc[mask, ['tenure', 'TotalCharges', 'MonthlyCharges']].head()

## 2. Очистка

In [ ]:
df = load_clean(PROJECT_ROOT / 'data' / 'raw' / 'telco_churn.csv')
print('После очистки:', df.shape)
print('Пропуски:', df.isna().sum().sum())
df.dtypes

## 3. Баланс классов

In [ ]:
vc = df['Churn'].value_counts(normalize=True)
print(vc)
fig, ax = plt.subplots(figsize=(5, 3))
sns.countplot(x='Churn', data=df, ax=ax)
ax.set_title('Распределение целевой переменной (0=остался, 1=ушёл)')
plt.show()

**Наблюдение:** классы несбалансированы — отток ~26.5%. Поэтому метрика accuracy будет вводить в заблуждение; будем смотреть на ROC-AUC, F1, precision/recall. В моделях используем `class_weight='balanced'`.

## 4. Распределения числовых признаков

In [ ]:
df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.histplot(data=df, x=col, hue='Churn', kde=True, ax=ax, element='step')
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Наблюдения:**
- `tenure`: ушедшие клиенты в основном новички (короткий tenure) — сильный сигнал для модели.
- `MonthlyCharges`: у ушедших сдвиг в сторону более высоких платежей.
- `TotalCharges` коррелирует с tenure (что логично).

## 5. Категориальные признаки vs Churn

In [ ]:
cat_cols = ['Contract', 'PaymentMethod', 'InternetService', 'OnlineSecurity', 'TechSupport', 'PaperlessBilling']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, cat_cols):
    rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    sns.barplot(x=rate.values, y=rate.index, ax=ax, orient='h')
    ax.set_title(f'Churn rate по {col}')
    ax.set_xlim(0, 0.6)
plt.tight_layout()
plt.show()

**Наблюдения:**
- Контракт `Month-to-month` — самый рискованный (~43% ухода) против `Two year` (~3%).
- Оплата `Electronic check` ассоциирована с высоким churn.
- Отсутствие `OnlineSecurity` и `TechSupport` повышает риск ухода.
- `PaperlessBilling=Yes` — выше churn.

## 6. Корреляция числовых признаков

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.show()

## Выводы EDA

1. Размер датасета: 7043 строк, 21 колонка. После удаления `customerID` остаётся 20 признаков.
2. Целевая переменная несбалансирована (~26.5% Churn=1). Метрики: ROC-AUC, F1, precision, recall. Для тренировки — `class_weight='balanced'`.
3. Пропусков нет (после приведения `TotalCharges` к числу и заполнения нулями для tenure=0).
4. Сильнейшие предикторы по EDA: `tenure`, `Contract`, `PaymentMethod`, `OnlineSecurity`, `TechSupport`, `MonthlyCharges`.
5. Числовые признаки → StandardScaler; категориальные → OneHotEncoder.
6. Сплит train/test: 80/20, стратифицированный по `Churn`, `random_state=42`.